# LDA (sklearn)

1. Set up and Sample Data

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# Sample documents
documents = [
    "Apple releases new iPhone with better camera",
    "Samsung unveils latest smartphone model",
    "Scientists discover water on Mars",
    "NASA plans mission to explore Mars",
    "New Android update improves battery life",
    "Apple announces MacBook with M2 chip"
]


2. Convert Text to Bag of Words

In [4]:
# Convert text to a term-document matrix
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(documents)

3. Apply LDA for Topic Modeling

In [5]:
# Create LDA model with 2 topics
lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(X)



,n_components,2
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


4. Display Topics and Top Words

In [6]:
# Display top words for each topic
def display_topics(model, feature_names, num_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx + 1}:")
        top_features = topic.argsort()[:-num_top_words - 1:-1]
        print("  " + ", ".join([feature_names[i] for i in top_features]))

# Print top 5 words per topic
display_topics(lda, vectorizer.get_feature_names_out(), 5)


Topic 1:
  mars, nasa, plans, mission, explore
Topic 2:
  new, apple, releases, iphone, camera


5. Check Topic Distribution Per Document (Optional)

In [7]:
# Get topic distribution for each document
doc_topics = lda.transform(X)
# these are probabilities representing the likelihood that the document belongs to each topic.
# This comes from the assumption in LDA that each document is a mix of topics, and each topic is a mix of words.

for i, topic_dist in enumerate(doc_topics):
    print(f"Document {i + 1} topics:", topic_dist)

Document 1 topics: [0.07566414 0.92433586]
Document 2 topics: [0.08948937 0.91051063]
Document 3 topics: [0.89770984 0.10229016]
Document 4 topics: [0.91470678 0.08529322]
Document 5 topics: [0.07615358 0.92384642]
Document 6 topics: [0.08876559 0.91123441]


In [12]:
import sys
print(sys.executable)
import pandas

c:\Program Files\Python313\python.exe


# sentence-transformers to generate embeddings

- Convert each email (or paragraph) into a dense vector.
- Use clustering (e.g. KMeans) to group similar messages.
- Clustering + Keyword Extraction (TF-IDF or keyburt (more semantic))

In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Sample emails (replace with yours)
emails = [
    "Please review the project proposal by Friday.",
    "The system will be down for maintenance at 2AM.",
    "Your invoice has been approved and processed.",
    "Let's set up a meeting to discuss the Q3 roadmap.",
    "There will be a team lunch on Thursday at noon.",
]

# Step 1: Embed emails
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(emails)

# Step 2: Cluster
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

# Step 3: Build dataframe
df = pd.DataFrame({'email': emails, 'cluster': labels})

# Step 4: Extract keywords for each cluster
topic_keywords = []
for i in range(n_clusters):
    cluster_emails = df[df['cluster'] == i]['email']
    if not cluster_emails.empty:
        tfidf = TfidfVectorizer(stop_words='english', max_features=5)
        tfidf_matrix = tfidf.fit_transform(cluster_emails)
        keywords = tfidf.get_feature_names_out()
        topic_keywords.append(", ".join(keywords))
    else:
        topic_keywords.append("No emails")

# Step 5: Map topic keywords back to clusters
df['topic'] = df['cluster'].map({i: topic_keywords[i] for i in range(n_clusters)})

print(df)


                                               email  cluster  \
0      Please review the project proposal by Friday.        2   
1    The system will be down for maintenance at 2AM.        0   
2      Your invoice has been approved and processed.        0   
3  Let's set up a meeting to discuss the Q3 roadmap.        1   
4    There will be a team lunch on Thursday at noon.        0   

                                        topic  
0           friday, project, proposal, review  
1  2am, approved, invoice, lunch, maintenance  
2  2am, approved, invoice, lunch, maintenance  
3          discuss, let, meeting, q3, roadmap  
4  2am, approved, invoice, lunch, maintenance  


keyburt example for keyword extraction

In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from keybert import KeyBERT
import pandas as pd

# === Step 1: Your email content ===
emails = [
    "Please review the project proposal by Friday.",
    "The system will be down for maintenance at 2AM.",
    "Your invoice has been approved and processed.",
    "Let's set up a meeting to discuss the Q3 roadmap.",
    "There will be a team lunch on Thursday at noon.",
    "We need to update the roadmap after the feedback session.",
    "Server restart scheduled for maintenance this weekend.",
    "Kindly approve the attached invoice for reimbursement.",
    "Project kickoff meeting is on Monday at 10AM.",
]

# === Step 2: Sentence embeddings and clustering ===
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(emails)

n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

df = pd.DataFrame({'email': emails, 'cluster': labels})

# === Step 3: Use KeyBERT to extract keywords per cluster ===
kw_model = KeyBERT(model=model)
cluster_keywords = {}

for cluster_id in range(n_clusters):
    # Combine all emails in the cluster into one block of text
    cluster_texts = df[df["cluster"] == cluster_id]["email"].tolist()
    combined_text = " ".join(cluster_texts)
    
    # Extract top 3 semantic keywords using KeyBERT
    keywords = kw_model.extract_keywords(combined_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=3)
    cluster_keywords[cluster_id] = ", ".join([kw[0] for kw in keywords])

# === Step 4: Map topic keywords to each row ===
df["topic_keywords"] = df["cluster"].map(cluster_keywords)

# === Step 5: Group and display ===
summary = df.groupby("cluster").agg({
    "topic_keywords": "first",
    "email": list
}).reset_index()

# Pretty print
for _, row in summary.iterrows():
    print(f"\n Cluster {row['cluster']} — Keywords: {row['topic_keywords']}")
    for email in row['email']:
        print(f"   - {email}")



 Cluster 0 — Keywords: q3 roadmap, update roadmap, roadmap need
   - Let's set up a meeting to discuss the Q3 roadmap.
   - We need to update the roadmap after the feedback session.
   - Project kickoff meeting is on Monday at 10AM.

 Cluster 1 — Keywords: scheduled maintenance, maintenance 2am, maintenance weekend
   - The system will be down for maintenance at 2AM.
   - Your invoice has been approved and processed.
   - There will be a team lunch on Thursday at noon.
   - Server restart scheduled for maintenance this weekend.

 Cluster 2 — Keywords: project proposal, invoice reimbursement, reimbursement
   - Please review the project proposal by Friday.
   - Kindly approve the attached invoice for reimbursement.


# Bertopic: BERT embeddings for context-aware

In [23]:
from bertopic import BERTopic
print("BERTopic imported successfully!")


BERTopic imported successfully!


In [27]:
from bertopic import BERTopic
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

emails = [
    "Please review the project proposal by Friday.",
    "The system will be down for maintenance at 2AM.",
    "Your invoice has been approved and processed.",
    "Let's set up a meeting to discuss the Q3 roadmap.",
    "There will be a team lunch on Thursday at noon.",
    "We need to update the roadmap after the feedback session.",
    "Server restart scheduled for maintenance this weekend.",
    "Kindly approve the attached invoice for reimbursement.",
    "Project kickoff meeting is on Monday at 10AM.",
]

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(emails)
kmeans = KMeans(n_clusters=3, random_state=42)

topic_model = BERTopic(hdbscan_model=kmeans, embedding_model=embedding_model, umap_model=None)
topics, probabilities = topic_model.fit_transform(emails, embeddings)

topic_info = topic_model.get_topic_info()

print("Topics found:\n")
for _, row in topic_info.iterrows():
    topic_num = row['Topic']
    if topic_num == -1:
        # Skip outliers
        continue
    docs_in_topic = row['Count']
    keywords = topic_model.get_topic(topic_num)
    # Extract top 5 keywords only
    top_keywords = ", ".join([word for word, _ in keywords[:5]])
    print(f"Topic {topic_num} ({docs_in_topic} documents): {top_keywords}")


Topics found:

Topic 0 (4 documents): the, project, meeting, to, roadmap
Topic 1 (3 documents): will, maintenance, be, at, for
Topic 2 (2 documents): invoice, kindly, approve, processed, reimbursement
